# Notebook 04 — Within-Cluster Anomaly Detection

## Goal
Flag wells that deviate significantly from their **cluster peers** — not from the global dataset.
This is critical for the resource-allocation framing: a well with an unusually high groundwater
level compared to its regional neighbours is a candidate for priority testing.

### Algorithm: Isolation Forest
- Builds an ensemble of random trees; anomalies are isolated with fewer splits
- Runs **independently per DBSCAN cluster** — so each well is judged against its local context
- Contamination parameter set to 5%: expects ~1-in-20 wells to be anomalous per cluster

### Key Diagnostic
The **anomaly vs. contamination heatmap** validates whether flagged wells actually have
higher contamination rates — if they do, the anomaly signal is geophysically meaningful.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

from src.anomaly import run_anomaly_pipeline

PROCESSED_DIR = '../data/processed'
FIGURES_DIR   = '../figures'

## Step 1 — Run Anomaly Detection Pipeline

In [ ]:
df, top_anomalies = run_anomaly_pipeline(PROCESSED_DIR, figures_dir=FIGURES_DIR)

print(f'Total wells scored : {df["anomaly_score"].notna().sum():,}')
print(f'Anomalies flagged  : {df["is_anomaly"].sum():,}')
print(f'Anomaly rate       : {df["is_anomaly"].mean():.2%}')

## Step 2 — Top Anomalous Wells (Priority Testing Candidates)

In [ ]:
cols = ['station','state','district','cluster_dbscan','anomaly_score','is_contaminated']
cols = [c for c in cols if c in top_anomalies.columns]
print(top_anomalies[cols].head(20).to_string(index=False))

## Step 3 — View Saved Figures

In [ ]:
from IPython.display import Image, display
import os

for fname in [
    'anomaly_score_distribution.png',
    'anomaly_map.png',
    'anomaly_vs_contamination.png',
]:
    fpath = os.path.join(FIGURES_DIR, fname)
    if os.path.exists(fpath):
        print(f'\n--- {fname} ---')
        display(Image(fpath))